## Wikipedia RfC Baseline Analysis

Run inside PAWS: https://wikitech.wikimedia.org/wiki/PAWS

**Queries:**
- Bot account history (verification superset)
- Q2: RfC volume over time (new pages per year)
- Q1: Participant tenure — RfC
- Q4: Participant tenure — RfA (comparison)
- Q3: Policy page edit activity, split by reverted / unreverted
- Bonus: Participation per RfC page

**Notes:**
- `page_namespace = 4` = Wikipedia: namespace on enwiki only
- Bot filter: `LEFT JOIN user_groups` on `ug_group = 'bot'` (current flag)
- Revert detection: `mw-reverted` change tag only (`rev_sha1` not available in replicas; pre-2021 reverts undercounted)
- All outputs go to `~/rfc-analysis/data/`

In [ ]:
import wmpaws
import pandas as pd
from pathlib import Path

OUT = Path.home() / 'rfc-analysis' / 'data'
OUT.mkdir(parents=True, exist_ok=True)
WIKI = 'enwiki'
print(f'Output folder: {OUT}')

## Bot accounts (verification superset)

Query all accounts that ever held the bot flag via the `logging` table.
Save as `bot_accounts_ever.csv` for spot-checking against tenure/participation results.
Primary bot filter in queries below uses `user_groups` (current flag) — this cell
lets you verify if any deflagged bots are slipping through.

In [ ]:
df_bots = wmpaws.run_sql("""
SELECT
    a.actor_user                        AS user_id,
    l.log_title                         AS username,
    MIN(l.log_timestamp)                AS first_bot_flag,
    MAX(l.log_timestamp)                AS last_rights_change
FROM logging l
LEFT JOIN actor a ON a.actor_name = l.log_title
WHERE l.log_type      = 'rights'
  AND l.log_action    = 'rights'
  AND l.log_namespace = 2
  AND l.log_params    LIKE '%bot%'
GROUP BY 1, 2
ORDER BY first_bot_flag
""", WIKI)

df_bots.to_csv(OUT / 'bot_accounts_ever.csv', index=False)
BOT_USER_IDS = set(df_bots['user_id'].dropna().astype(int))
print(f'{len(df_bots):,} accounts ever flagged as bot')
df_bots.head(20)

## Q2 — RfC volume over time

Count new RfC pages opened per year/month (by first edit date, not last-touched date).
Counting by `MIN(rev_timestamp)` per page avoids inflation from archival bots
touching old pages in later years.

In [ ]:
df_volume = wmpaws.run_sql("""
SELECT
    LEFT(first_edit, 6)        AS ym,
    COUNT(*)                   AS new_pages,
    SUM(total_edits)           AS total_edits
FROM (
    SELECT
        r.rev_page,
        MIN(r.rev_timestamp)   AS first_edit,
        COUNT(*)               AS total_edits
    FROM revision r
    JOIN page p ON r.rev_page = p.page_id
    WHERE p.page_namespace = 4
      AND p.page_title LIKE 'Requests_for_comment/%'
      AND r.rev_timestamp >= '20060101000000'
    GROUP BY r.rev_page
) sub
GROUP BY 1
ORDER BY 1
""", WIKI)

df_volume['year']  = df_volume['ym'].str[:4].astype(int)
df_volume['month'] = df_volume['ym'].str[4:6].astype(int)
df_volume.to_csv(OUT / 'rfc_volume_by_month.csv', index=False)
print(f'{len(df_volume)} months')
df_volume.groupby('year')[['new_pages', 'total_edits']].sum()

## Q1 — Participant tenure in RfC discussions

For each non-bot edit to an RfC subpage, compute the editor's account age at time of edit.
Split into **reverted** vs **unreverted** edits — reverted edits likely represent
rejected contributions or maintenance noise; unreverted edits are accepted participation.

Revert detection: `mw-reverted` change tag (available 2021+). Pre-2021 reverts are not detected
— all pre-2021 edits appear as `unreverted`.

In [ ]:
df_tenure_rfc = wmpaws.run_sql("""
SELECT
    LEFT(r.rev_timestamp, 4)                                AS yr,
    DATEDIFF(
        STR_TO_DATE(r.rev_timestamp,     '%Y%m%d%H%i%S'),
        STR_TO_DATE(u.user_registration, '%Y%m%d%H%i%S')
    )                                                       AS tenure_days,
    a.actor_name                                            AS username,
    CASE
        WHEN ctd.ctd_name = 'mw-reverted' THEN 'reverted'
        ELSE 'unreverted'
    END                                                     AS edit_status
FROM revision r
JOIN page        p   ON r.rev_page    = p.page_id
JOIN actor       a   ON r.rev_actor   = a.actor_id
JOIN `user`      u   ON a.actor_user  = u.user_id
LEFT JOIN user_groups ug  ON a.actor_user = ug.ug_user AND ug.ug_group = 'bot'
LEFT JOIN change_tag ct   ON r.rev_id = ct.ct_rev_id
LEFT JOIN change_tag_def ctd ON ct.ct_tag_id = ctd.ctd_id AND ctd.ctd_name = 'mw-reverted'
WHERE p.page_namespace = 4
  AND p.page_title LIKE 'Requests_for_comment/%'
  AND r.rev_timestamp >= '20060101000000'
  AND r.rev_actor IS NOT NULL
  AND a.actor_user IS NOT NULL
  AND u.user_registration IS NOT NULL
  AND ug.ug_user IS NULL
""", WIKI)

df_tenure_rfc.to_csv(OUT / 'rfc_tenure_raw.csv', index=False)
print(f'{len(df_tenure_rfc):,} rows')
df_tenure_rfc.groupby('edit_status')['tenure_days'].describe().round(1)

In [ ]:
tenure_by_year = (
    df_tenure_rfc
    .query("edit_status == 'unreverted'")
    .groupby('yr')['tenure_days']
    .agg(
        median='median',
        mean='mean',
        count='count',
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75),
    )
    .reset_index()
    .rename(columns={'yr': 'year'})
)
tenure_by_year['mean'] = tenure_by_year['mean'].round(0).astype(int)
tenure_by_year.to_csv(OUT / 'rfc_tenure_by_year.csv', index=False)
tenure_by_year

## Q4 — RfA participant tenure (comparison)

Same query as Q1 but for `Requests_for_adminship/%` pages.
Are RfA voters more experienced than RfC participants? Has the gap changed over time?

In [ ]:
df_tenure_rfa = wmpaws.run_sql("""
SELECT
    LEFT(r.rev_timestamp, 4)                                AS yr,
    DATEDIFF(
        STR_TO_DATE(r.rev_timestamp,     '%Y%m%d%H%i%S'),
        STR_TO_DATE(u.user_registration, '%Y%m%d%H%i%S')
    )                                                       AS tenure_days,
    a.actor_name                                            AS username,
    CASE
        WHEN ctd.ctd_name = 'mw-reverted' THEN 'reverted'
        ELSE 'unreverted'
    END                                                     AS edit_status
FROM revision r
JOIN page        p   ON r.rev_page    = p.page_id
JOIN actor       a   ON r.rev_actor   = a.actor_id
JOIN `user`      u   ON a.actor_user  = u.user_id
LEFT JOIN user_groups ug  ON a.actor_user = ug.ug_user AND ug.ug_group = 'bot'
LEFT JOIN change_tag ct   ON r.rev_id = ct.ct_rev_id
LEFT JOIN change_tag_def ctd ON ct.ct_tag_id = ctd.ctd_id AND ctd.ctd_name = 'mw-reverted'
WHERE p.page_namespace = 4
  AND p.page_title LIKE 'Requests_for_adminship/%'
  AND r.rev_timestamp >= '20060101000000'
  AND r.rev_actor IS NOT NULL
  AND a.actor_user IS NOT NULL
  AND u.user_registration IS NOT NULL
  AND ug.ug_user IS NULL
""", WIKI)

df_tenure_rfa.to_csv(OUT / 'rfa_tenure_raw.csv', index=False)
print(f'{len(df_tenure_rfa):,} rows')
df_tenure_rfa.groupby('edit_status')['tenure_days'].describe().round(1)

In [ ]:
# Side-by-side: unreverted edits only
comparison = pd.DataFrame({
    'process': ['RfC', 'RfA'],
    'n': [
        len(df_tenure_rfc.query("edit_status == 'unreverted'")),
        len(df_tenure_rfa.query("edit_status == 'unreverted'")),
    ],
    'median_days': [
        df_tenure_rfc.query("edit_status == 'unreverted'")['tenure_days'].median(),
        df_tenure_rfa.query("edit_status == 'unreverted'")['tenure_days'].median(),
    ],
    'p25_days': [
        df_tenure_rfc.query("edit_status == 'unreverted'")['tenure_days'].quantile(0.25),
        df_tenure_rfa.query("edit_status == 'unreverted'")['tenure_days'].quantile(0.25),
    ],
    'p75_days': [
        df_tenure_rfc.query("edit_status == 'unreverted'")['tenure_days'].quantile(0.75),
        df_tenure_rfa.query("edit_status == 'unreverted'")['tenure_days'].quantile(0.75),
    ],
})
comparison['median_years'] = (comparison['median_days'] / 365).round(1)
comparison.to_csv(OUT / 'rfc_rfa_tenure_comparison.csv', index=False)
comparison

## Q3 — Policy page edit activity, reverted vs unreverted

Track edits per year on core policy pages, split by whether the edit was reverted.
High revert rate = contested policy. Unreverted edit rate = stable policy evolution.

In [ ]:
POLICY_PAGES = [
    'Requests_for_comment',
    'Consensus',
    'Policies_and_guidelines',
    'Blocking_policy',
    'Banning_policy',
    'Biographies_of_living_persons',
    'Neutral_point_of_view',
    'Verifiability',
    'No_original_research',
    'Civility',
    'Arbitration/Policy',
]
pages_str = ', '.join(f"'{p}'" for p in POLICY_PAGES)

df_policy_edits = wmpaws.run_sql(f"""
SELECT
    p.page_title                        AS page_title,
    LEFT(r.rev_timestamp, 4)            AS yr,
    COUNT(*)                            AS total_edits,
    COUNT(DISTINCT a.actor_name)        AS distinct_editors,
    SUM(CASE WHEN ctd.ctd_name = 'mw-reverted' THEN 1 ELSE 0 END) AS reverted_edits,
    SUM(CASE WHEN ctd.ctd_name = 'mw-reverted' THEN 0 ELSE 1 END) AS unreverted_edits
FROM revision r
JOIN page        p   ON r.rev_page   = p.page_id
JOIN actor       a   ON r.rev_actor  = a.actor_id
LEFT JOIN user_groups ug  ON a.actor_user = ug.ug_user AND ug.ug_group = 'bot'
LEFT JOIN change_tag ct   ON r.rev_id = ct.ct_rev_id
LEFT JOIN change_tag_def ctd ON ct.ct_tag_id = ctd.ctd_id AND ctd.ctd_name = 'mw-reverted'
WHERE p.page_namespace = 4
  AND p.page_title IN ({pages_str})
  AND r.rev_timestamp >= '20040101000000'
  AND ug.ug_user IS NULL
GROUP BY 1, 2
ORDER BY 1, 2
""", WIKI)

df_policy_edits['revert_rate'] = (
    df_policy_edits['reverted_edits'] / df_policy_edits['total_edits']
).round(3)
df_policy_edits.to_csv(OUT / 'policy_page_edits_by_year.csv', index=False)
print(f'{len(df_policy_edits):,} rows')
df_policy_edits.pivot_table(index='page_title', columns='yr',
                             values='revert_rate', fill_value=0).round(2)

## Bonus — Participation per RfC page

For each RfC subpage: distinct editors, total edits, duration, and revert rate.
Gives the distribution of how contested / well-attended a typical RfC is.

In [ ]:
df_participation = wmpaws.run_sql("""
SELECT
    p.page_title                        AS rfc_page,
    LEFT(MIN(r.rev_timestamp), 4)       AS yr,
    COUNT(DISTINCT a.actor_name)        AS distinct_editors,
    COUNT(*)                            AS total_edits,
    SUM(CASE WHEN ctd.ctd_name = 'mw-reverted' THEN 1 ELSE 0 END) AS reverted_edits,
    MIN(r.rev_timestamp)                AS first_edit,
    MAX(r.rev_timestamp)                AS last_edit,
    DATEDIFF(
        STR_TO_DATE(MAX(r.rev_timestamp), '%Y%m%d%H%i%S'),
        STR_TO_DATE(MIN(r.rev_timestamp), '%Y%m%d%H%i%S')
    )                                   AS duration_days
FROM revision r
JOIN page        p   ON r.rev_page  = p.page_id
JOIN actor       a   ON r.rev_actor = a.actor_id
LEFT JOIN user_groups ug  ON a.actor_user = ug.ug_user AND ug.ug_group = 'bot'
LEFT JOIN change_tag ct   ON r.rev_id = ct.ct_rev_id
LEFT JOIN change_tag_def ctd ON ct.ct_tag_id = ctd.ctd_id AND ctd.ctd_name = 'mw-reverted'
WHERE p.page_namespace = 4
  AND p.page_title LIKE 'Requests_for_comment/%'
  AND r.rev_timestamp >= '20060101000000'
  AND r.rev_actor IS NOT NULL
  AND ug.ug_user IS NULL
GROUP BY p.page_id
HAVING COUNT(*) >= 2
ORDER BY 2, 1
""", WIKI)

df_participation['revert_rate'] = (
    df_participation['reverted_edits'] / df_participation['total_edits']
).round(3)
df_participation.to_csv(OUT / 'rfc_participation_per_page.csv', index=False)
print(f'{len(df_participation):,} RfC pages')
df_participation[['distinct_editors','total_edits','duration_days','revert_rate']].describe().round(2)

## Summary

In [ ]:
rfc_unrev = df_tenure_rfc.query("edit_status == 'unreverted'")
rfa_unrev = df_tenure_rfa.query("edit_status == 'unreverted'")

lines = [
    '# RfC PAWS Baseline — Summary',
    f'Bot accounts ever flagged:        {len(df_bots):,}',
    f'RfC pages (new, 2006+):           {df_volume["new_pages"].sum():,}',
    f'Year range:                       {df_volume["year"].min()} – {df_volume["year"].max()}',
    f'Median tenure RfC unreverted:     {rfc_unrev["tenure_days"].median():.0f} days',
    f'Median tenure RfA unreverted:     {rfa_unrev["tenure_days"].median():.0f} days',
    f'Median editors per RfC:           {df_participation["distinct_editors"].median():.1f}',
    f'Median RfC duration:              {df_participation["duration_days"].median():.1f} days',
    f'Median RfC revert rate:           {df_participation["revert_rate"].median():.1%}',
]
summary = '\n'.join(lines)
print(summary)
(OUT / 'summary.txt').write_text(summary)